In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)

In [67]:
data.head(1)

,state_open,has_title,has_body,is_locked,is_closed,has_assignees,has_labels,author_association,user_type,repo_size,repo_stargazer_count,repo_watcher_count,repo_has_issue,repo_has_projects,repo_has_downloads,repo_has_wiki,repo_has_pages,repo_has_discussions,repo_fork_count,merged
0,1,1,1,0,0,0,1,0,0,1335380,111257,111257,1,1,1,0,0,0,31553,0


In [68]:
### checks initial initial class distribution
data['merged'].value_counts()

merged
1    19458
0    10542
Name: count, dtype: int64

In [3]:

X = data.iloc[:,:-1].values
y = data['merged'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30,
                                                    random_state=15, stratify=y)

In [70]:

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3,5,7,10]
}

scoring = 'f1'

clf = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, scoring=scoring, cv=10)

In [71]:
clf.fit(X_train, y_train)


print("Mejor combinación de parámetros:")
print(clf.best_params_)
 
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

Mejor combinación de parámetros:
{'criterion': 'gini', 'max_depth': 10}
              precision    recall  f1-score   support

           0       0.94      0.66      0.78      3163
           1       0.84      0.98      0.90      5837

    accuracy                           0.87      9000
   macro avg       0.89      0.82      0.84      9000
weighted avg       0.88      0.87      0.86      9000



# PREPARING MULTIPLE MODELS

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

In [5]:
### DUMMY (baseline)

param_grid_dm = {
    'strategy': ['most_frequent', 'stratified', 'uniform']
}

scoring = 'f1'

clf_dm = GridSearchCV(DummyClassifier(), param_grid=param_grid_dm, scoring=scoring, cv=10)

In [ ]:
### DECISION TREE

param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

scoring = 'f1'

clf_dt = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid_dt, scoring=scoring, cv=10, n_jobs=-1)

In [7]:
### RANDOM FOREST

param_grid_rf = {
    'n_estimators': [50],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'criterion': ['gini', 'entropy']
}

clf_rf = GridSearchCV(RandomForestClassifier(), param_grid=param_grid_rf, scoring='f1', cv=10, n_jobs=-1)


In [ ]:
### SUPPORT VECTOR CLASSIFIER

param_grid_svc = {
    'C': [1],          
    'kernel': ['rbf'],
}

clf_svc = GridSearchCV(SVC(), param_grid=param_grid_svc, scoring='f1', cv=10, n_jobs=-1)


In [9]:
### GAUSSIANNB

param_grid_nb = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
}

clf_nb = GridSearchCV(GaussianNB(), param_grid=param_grid_nb, scoring='f1', cv=10, n_jobs=-1)


In [ ]:
### K-NEIGHBORS

param_grid_knn = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

clf_knn = GridSearchCV(KNeighborsClassifier(), param_grid=param_grid_knn, scoring='f1', cv=10, n_jobs=-1)


In [30]:
classifiers = {
    "Base dummy": clf_dm,
    "Decision Tree": clf_dt, 
    "Random Forest": clf_rf, 
    "Support Vector Classifier": clf_svc, 
    "GaussianNB": clf_nb, 
    "K-neighbors": clf_knn
}

In [31]:
def train_with_multiples_models(classifiers: dict, X_train, y_train, X_test, y_test):
    for name, clf in classifiers.items():
        print(':::::::::::::::::::::::::::::::::::::::::')
        print(f'Current classifier: {name}')
        print('Training..')
        clf.fit(X_train, y_train)

        print("Mejor combinación de parámetros:")
        print(clf.best_params_)
        
        y_pred = clf.predict(X_test)
        print(classification_report(y_test, y_pred))

# DOING SOME BALANCE

In [72]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)

In [16]:
data['merged'].value_counts()

merged
1    19458
0    10542
Name: count, dtype: int64

In [17]:
X = data.drop(columns=['merged'])
y = data['merged']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30, random_state=15, stratify=y)


### UNDERSAMPLE

In [34]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print("Balanced class distribution (undersampling):")
print(y_resampled.value_counts())

Balanced class distribution (undersampling):
merged
0    7379
1    7379
Name: count, dtype: int64


In [35]:
train_with_multiples_models(classifiers=classifiers, X_train=X_resampled, y_train=y_resampled, X_test=X_test, y_test=y_test)

:::::::::::::::::::::::::::::::::::::::::
Current classifier: Base dummy
Training..
Mejor combinación de parámetros:
{'strategy': 'stratified'}
              precision    recall  f1-score   support

           0       0.35      0.50      0.41      3163
           1       0.65      0.50      0.56      5837

    accuracy                           0.50      9000
   macro avg       0.50      0.50      0.49      9000
weighted avg       0.54      0.50      0.51      9000

:::::::::::::::::::::::::::::::::::::::::
Current classifier: Decision Tree
Training..
Mejor combinación de parámetros:
{'criterion': 'gini', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
              precision    recall  f1-score   support

           0       0.85      0.71      0.78      3163
           1       0.86      0.93      0.89      5837

    accuracy                           0.86      9000
   macro avg       0.85      0.82      0.83      9000
weighted avg       0.85      

### OVERSAMPLE

In [19]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

print("Balanced class distribution (oversampling):")
print(y_resampled.value_counts())

Balanced class distribution (oversampling):
merged
1    13621
0    13621
Name: count, dtype: int64


In [32]:
train_with_multiples_models(classifiers=classifiers, X_train=X_resampled, y_train=y_resampled, X_test=X_test, y_test=y_test)

:::::::::::::::::::::::::::::::::::::::::
Current classifier: Base dummy
Training..
Mejor combinación de parámetros:
{'strategy': 'stratified'}
              precision    recall  f1-score   support

           0       0.35      0.50      0.41      3163
           1       0.65      0.50      0.56      5837

    accuracy                           0.50      9000
   macro avg       0.50      0.50      0.49      9000
weighted avg       0.54      0.50      0.51      9000

:::::::::::::::::::::::::::::::::::::::::
Current classifier: Decision Tree
Training..
Mejor combinación de parámetros:
{'criterion': 'entropy', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 5}
              precision    recall  f1-score   support

           0       0.85      0.72      0.78      3163
           1       0.86      0.93      0.89      5837

    accuracy                           0.85      9000
   macro avg       0.85      0.82      0.83      9000
weighted avg       0.85   

### Conclusion
Given the metrics and testing, we decided to use f1-score as the deciding factor when choosing a model, we want a good balance between having low amounts of false positives and false negatives.
We also considered how to balance the data do have better results, given the amount of data we currently have we considered oversampling as a better technique as it allow us to keep all the original data while addressing class imbalance.

The best models corresponds to a random forest, with the closest alternative being decision tree, we consider random forest as a better representation of the problem as it reduces overfitting a problem that its often associated to a decision tree.

Best model:
- Random Forest
- criterion: entropy
- max_depth: 10
- max_features: sqrt
- min_samples_leaf: 2
- min_samples_split: 2
- n_estimators: 50


In [36]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=50, criterion='entropy', max_depth=10, max_features='sqrt', min_samples_leaf=2, min_samples_split=2)

clf.fit(X_resampled, y_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.72      0.78      3163
           1       0.86      0.93      0.89      5837

    accuracy                           0.86      9000
   macro avg       0.85      0.82      0.83      9000
weighted avg       0.85      0.86      0.85      9000

